# Imports
Functions are kept in a seperate file to not clutter this report uneccessarily. Most imports are in functions.py

In [ ]:
from functions import *

# Read and view data
Setting path variables to the training and test data.

Makes a dataset for training, validation and testing.

Show general information about the data. Shape and the different classes. Also confirms that 20% of the total data is for testing. The remaining 80% of the data consists of training and validation data.

In [ ]:
train_path = "Data/train/"
test_path = "Data/test/"

# Create a training set
print("Train dataset:")
train_ds, class_names = load_images_from_dir(train_path, (48, 48), "training")

# Create a training set
print("\nValidation dataset:")
val_ds, _ = load_images_from_dir(train_path, (48, 48), "validation")

# Create a test set
print("\nTest dataset:")
test_ds, _ = load_images_from_dir(test_path, (48, 48), "")

# Check classes
print(f"\nClasses: {class_names}")

# View class names for training set
train_df = return_class_count_df(train_path)
print(f"\nTraining/Validation classes: \n{train_df}")

# View class names for test set
test_df = return_class_count_df(test_path)
print(f"\nTest classes: \n{test_df}")

# Check if the train and test data are split 80/20
print(f"\nPercentage of data in train/val dataset: {count_files(train_path) / (count_files(train_path) + count_files(test_path)):0.2f}")
print(f"Percentage of data in test dataset: {count_files(test_path) / (count_files(train_path) + count_files(test_path)):0.2f}")

Plotting n number of images to get an idea of what the data consists of.

In [ ]:
plot_n_images(train_ds, class_names, 5)

# Model 1 - Baseline
This model does not contain any forms of regularization. Exclusively different layers and activations.

This model acts as the baseline.

In [ ]:
baseline_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPool2D(pool_size=(2, 2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPool2D(pool_size=(2, 2)),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),

    layers.Dense(7, activation="softmax")
]

model_baseline = create_model(baseline_layers)

Returns the model and history after training.

In [ ]:
model1, history1 = train_model(model_baseline, train_ds, val_ds, 20, None)

# Model 1 conclusion
There are clear patterns of overfitting. Training and validation data are diverging rapidly after roughly epoch 2-3.

In [ ]:
plot_history_data(history1.history, "Model 1 - Baseline")

# Augmentation test
Adding augmentation to the training data to try improve the model. Adds randomness and explains the jagged validation curves. In general the model is learning. Accuracy is lower and loss higher with the augmentation but the model is more stable. No clear signs of overfitting now. An attempt to improve this model will be made by adding GlobalAveragePooling2D.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

In [ ]:
train_ds_augmented = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
augment_model = create_model(baseline_layers)

In [ ]:
model2, history2 = train_model(augment_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history2.history, "Model 2 - Augmentation")

# GlobalAveragePooling2D Test - Keeping augmentation

In [ ]:
global_avg_pool_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPool2D(pool_size=(2, 2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), activation="relu", padding="same"),
    layers.MaxPool2D(pool_size=(2, 2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64, activation="relu"),
    layers.Dense(7, activation="softmax")
]

In [ ]:
global_avg_pool_model = create_model(model1)

In [ ]:
model3, history3 = train_model(global_avg_pool_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history3.history, "Model 3 - GlobalAveragePooling2D")

# BatchNormalization - Keeping GAP and AUG
This combination made the model worse. Training loss has plateued and validation loss is jumping a lot. Training accuracy is increasing steadily and smoothly but the validation accuracy is very jagged. BN will be removed and a new Conv2D added instead.

In [ ]:
batch_normal_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    layers.Dense(7, activation="softmax")
]

In [ ]:
batch_normal_model = create_model(batch_normal_layers)

In [ ]:
model4, history4 = train_model(batch_normal_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history4.history, "Model 4 - Batch Normalization")

# Extra 64 Conv2D Layer - Keeping GAP and AUG
These results look significantly better as the graphs are much closer to each other and follow each other closely. It appears that a plateu is approaching around epoch 20.

In [ ]:
extra_layer_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64),
    layers.Activation("relu"),

    layers.Dense(7, activation="softmax")
]

In [ ]:
extra_layer_model = create_model(extra_layer_layers)

In [ ]:
model5, history5 = train_model(extra_layer_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history5.history, "Model 5 - Extra Layer")

# Less aggressive augmentation - Keeping extra 64 Conv2D, GAP and AUG
This model did nothing to improve the model nor did it worsen it. Changing the last Conv2D layer to 128 instead of 64 makes sense to test as the resolution of the images reduces by half.

In [ ]:
augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.05),
    layers.RandomContrast(0.05)
])

train_ds_less_augmented = train_ds.map(
    lambda x, y: (augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
less_augment_model = create_model(extra_layer_layers)
model6, history6 = train_model(less_augment_model, train_ds_less_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history6.history, "Model 6 - Less Augment")

# Changing 64Conv2D to 128Conv2D - Keeping GAP and AUG - Removed decrease in Augment
Tiny improvement but not significant. Will add dropout(0.3).

In [ ]:
extra_layer_2_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=128, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64),
    layers.Activation("relu"),

    layers.Dense(7, activation="softmax")
]

In [ ]:
neuron_128_model = create_model(extra_layer_2_layers)
model7, history7 = train_model(neuron_128_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history7.history, "Model 7 - 64 to 128 layer")

# Dropout - Same as before
No improvement

In [ ]:
dropout_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D( filters=128, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64),
    layers.Activation("relu"),

    layers.Dropout(0.3),

    layers.Dense(7, activation="softmax")
]

In [ ]:
dropout_model = create_model(dropout_layers)
model8, history8 = train_model(dropout_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history8.history, "Model 8 - 0.3 dropout")

# Adding 256 layer

In [ ]:
added_256_layer_layers = [
    layers.Input(shape=(48, 48, 1)),

    layers.Conv2D(filters=32, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=128, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.Conv2D(filters=256, kernel_size=(3, 3), padding="same"),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=(2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(64),
    layers.Activation("relu"),

    layers.Dense(7, activation="softmax")
]

In [ ]:
added_256_layer_model = create_model(added_256_layer_layers)
model9, history9 = train_model(added_256_layer_model, train_ds_augmented, val_ds, 20, None)

In [ ]:
plot_history_data(history9.history, "Model 9 - 256 layer")

# Model comparison
All model's histories are compared to determine which model to use based on val_accuracy

In [ ]:
models = [model1, model2, model3, model4, model5,
          model6, model7, model8, model9]

histories = [history1, history2, history3, history4, history5,
             history6, history7, history8, history9]

results = []

for i, history in enumerate(histories, start=1):

    hist = history.history

    best_val_acc = max(hist["val_accuracy"])
    best_val_loss = min(hist["val_loss"])

    best_epoch = hist["val_accuracy"].index(best_val_acc) + 1

    results.append({
        "Model": f"Model{i}",
        "Final Train Acc": hist["accuracy"][-1],
        "Final Val Acc": hist["val_accuracy"][-1],
        "Best Val Acc": best_val_acc,
        "Best Val Loss": best_val_loss,
        "Best Epoch": best_epoch
    })

df = pd.DataFrame(results)

df = df.sort_values(
    by="Best Val Acc",
    ascending=False
)

print(df)

In [ ]:
plt.figure(figsize=(15, 9))

count = 1
for i in histories:
    plt.plot(i.history["accuracy"], label=f"model {count}")
    count += 1

plt.legend()
plt.show()
